# Vast.ai — MLP and ST-GNN with stratified 64/16/20 splitting

Upload a full-data CSV/ZIP, or the existing train/test ZIP, under `/workspace/upload`. Existing train/test files are combined into one pool and their old split is replaced. First split the full pool into **80% development / 20% test**, then split development into **80% train / 20% validation**. Final proportions are approximately **64% train / 16% validation / 20% test**, allowing integer rounding. Both splits use `stratify`; every partition is checked for **Severity 1, 2, 3, and 4** before training. Missing classes cause an error, never fabricated samples.

Use `RANDOM_STATE=42` to reproduce this single split. MLP and ST-GNN share the exact same partitions. Preprocessing is fitted on training rows, and early stopping/threshold tuning use validation. The existing ST-GNN remains transductive: all partition features participate in one graph, while training loss uses training seed nodes. This replaces the old held-out-test protocol and is not directly comparable with old fixed-test results.

Select the Vast PyTorch/Jupyter template and install matching PyTorch Geometric extensions as described below. No graph-node cap is applied. A full run may need substantial system RAM and time. Split counts and row IDs are saved in the results folder.


## Vast terminal setup (run once before this notebook)

Use the Python environment of the Jupyter kernel. The PyTorch template should already contain CUDA-enabled Torch. Check `python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"` first, then run:

```bash
python -m pip install 'scikit-learn>=1.4' pandas numpy scipy joblib torch_geometric
PYG_TORCH=$(python -c "import torch; v=torch.__version__.split('+')[0].split('.'); print(f'{v[0]}.{v[1]}.0')")
PYG_CUDA=$(python -c "import torch; print('cu'+torch.version.cuda.replace('.',''))")
python -m pip install pyg_lib torch_scatter torch_sparse -f "https://data.pyg.org/whl/torch-${PYG_TORCH}+${PYG_CUDA}.html"
```

Restart the Jupyter kernel after first-time package installation. Do not install CPU extension wheels on a CUDA template.

In [ ]:
import json, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,
                             precision_recall_curve,classification_report,confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

assert torch.cuda.is_available(), 'Rent a CUDA GPU and select the PyTorch Jupyter kernel.'
try:
    import pyg_lib
except ImportError:
    try: import torch_sparse
    except ImportError as exc: raise ImportError('NeighborLoader needs a matching pyg_lib or torch_sparse wheel. See setup instructions.') from exc
DEVICE=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,'CUDA:',torch.version.cuda)
INPUT_ROOT=Path('/workspace/upload')
OUT_DIR=Path('/workspace/dl_results')
OUT_DIR.mkdir(parents=True,exist_ok=True)
MLP_EPOCHS=100
MLP_PATIENCE=10
STGNN_EPOCHS=50
STGNN_PATIENCE=10
STGNN_K=5
MLP_BATCH_SIZE=1024
GNN_BATCH_SIZE=1024
TARGET = 'Severity'
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.20  # fraction of the 80% development pool
REQUIRED_SEVERITY_LEVELS = [1, 2, 3, 4]
MAX_CAT_ONEHOT = 50
USE_CLASS_WEIGHT = True
CLASS_WEIGHT_CAP = 2.5
DATETIME_COLS = ['Start_Time', 'End_Time', 'Weather_Timestamp']
BOOL_COLS = ['Amenity','Crossing','Give_Way','Junction','No_Exit','Railway','Station','Stop','Traffic_Signal']
GEO_COLS = ['Start_Lat', 'Start_Lng']
YELLOW_EXCLUDE = ['ID','Source','End_Lat','End_Lng','Country','Wind_Chill(F)',
                  'Precipitation(in)','Bump','Roundabout','Traffic_Calming','Turning_Loop']
METRICS = ['Accuracy','Macro_Precision','Macro_Recall','Macro_F1']

# Set this explicitly if your input location has more than one ZIP.
DATA_ZIP = None  # optional explicit ZIP path
FULL_CSV = None  # optional explicit full-data CSV path
CSV_MEMBER = None  # optional member in a ZIP containing multiple CSVs
TRAIN_CSV = None  # optional old train CSV; set together with TEST_CSV
TEST_CSV = None

## Load a full pool and create stratified train/validation/test partitions
Existing train/test files are recombined before the new split. You can also supply one full CSV or a ZIP containing one full CSV. Every level 1–4 must exist in the full pool and each resulting partition.


In [ ]:
def engineer_features(df):
    df = df.copy()
    df = df.drop(columns=YELLOW_EXCLUDE, errors="ignore")
    # End_Time occurs after an accident and must not define an input feature.
    df = df.drop(columns=["End_Time"], errors="ignore")
    for c in DATETIME_COLS:
        if c in df.columns: df[c] = pd.to_datetime(df[c], errors="coerce")
    if "Start_Time" in df.columns:
        df["Start_Hour"] = df["Start_Time"].dt.hour
        df["Start_Dayofweek"] = df["Start_Time"].dt.dayofweek
        df["Start_IsWeekend"] = (df["Start_Dayofweek"] >= 5).astype(int)
        df["Is_RushHour"] = df["Start_Hour"].isin([7,8,9,16,17,18]).astype(int)
    if "Weather_Timestamp" in df.columns:
        df["Weather_Hour"] = df["Weather_Timestamp"].dt.hour
    df = df.drop(columns=[c for c in DATETIME_COLS if c in df.columns], errors="ignore")
    for c in BOOL_COLS:
        if c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].map({"True":1,"False":0,True:1,False:0,"TRUE":1,"FALSE":0,"Y":1,"N":0})
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    if "Weather_Condition" in df.columns:
        df["Weather_Condition"] = (df["Weather_Condition"].fillna("Missing").astype(str).str.lower().str.strip())
        # Frequency/target encoding learns its vocabulary from each training fold only.
    for c in df.columns:
        if df[c].dtype == object: df[c] = df[c].fillna("Missing").astype(str)
    df = df.dropna(subset=[TARGET])  # keep rows with missing coordinates
    return df

In [ ]:
import hashlib, gc
from zipfile import ZipFile
from IPython.display import display

def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def combine_frames(frames):
    if any(TARGET not in frame for frame in frames):
        raise ValueError('Every input CSV must contain Severity labels.')
    if any(set(frame.columns) != set(frames[0].columns) for frame in frames[1:]):
        raise ValueError('Input CSV feature schemas differ.')
    return pd.concat(frames, ignore_index=True)

def locate_full_pool(search_root):
    # Explicit plain CSV paths take priority over directory auto-detection.
    if FULL_CSV is not None:
        if DATA_ZIP is not None or TRAIN_CSV is not None or TEST_CSV is not None:
            raise ValueError('Set only FULL_CSV, DATA_ZIP, or the TRAIN_CSV/TEST_CSV pair.')
        paths = [Path(FULL_CSV)]
    elif TRAIN_CSV is not None or TEST_CSV is not None:
        if DATA_ZIP is not None or TRAIN_CSV is None or TEST_CSV is None:
            raise ValueError('Set both TRAIN_CSV and TEST_CSV, without DATA_ZIP.')
        paths = [Path(TRAIN_CSV), Path(TEST_CSV)]
        if paths[0].resolve() == paths[1].resolve():
            raise ValueError('Train and test input paths must be different.')
    else:
        archives = [Path(DATA_ZIP)] if DATA_ZIP is not None else list(Path(search_root).rglob('*.zip'))
        if archives:
            if len(archives) != 1:
                raise ValueError(f'Found multiple ZIPs; set DATA_ZIP explicitly: {archives}')
            archive = archives[0]
            if not archive.is_file():
                raise FileNotFoundError(archive)
            with ZipFile(archive) as z:
                csvs = [s for s in z.namelist() if s.lower().endswith('.csv') and not s.startswith('__MACOSX/')]
                if CSV_MEMBER is not None:
                    if CSV_MEMBER not in csvs:
                        raise ValueError(f'CSV_MEMBER not found. Available: {csvs}')
                    selected = [CSV_MEMBER]
                elif len(csvs) == 1:
                    selected = csvs
                else:
                    trains = [s for s in csvs if 'train' in Path(s).name.lower()]
                    tests = [s for s in csvs if 'test' in Path(s).name.lower()]
                    if len(csvs) != 2 or len(trains) != 1 or len(tests) != 1 or trains[0] == tests[0]:
                        raise ValueError(f'Expected one full CSV or exactly one train/test pair; set CSV_MEMBER: {csvs}')
                    selected = [trains[0], tests[0]]
                frames, members = [], []
                for member in selected:
                    with z.open(member) as f:
                        frame = pd.read_csv(f, low_memory=False)
                    frames.append(frame)
                    info = z.getinfo(member)
                    members.append({'member': member, 'rows': len(frame),
                                    'crc32': f'{info.CRC:08x}', 'bytes': info.file_size})
            return combine_frames(frames), {'zip': str(archive), 'members': members,
                                            'combined_existing_splits': len(selected) == 2}
        csvs = list(Path(search_root).rglob('*.csv'))
        if len(csvs) == 1:
            paths = csvs
        else:
            trains = [p for p in csvs if 'train' in p.name.lower()]
            tests = [p for p in csvs if 'test' in p.name.lower()]
            if len(csvs) != 2 or len(trains) != 1 or len(tests) != 1 or trains[0] == tests[0]:
                raise FileNotFoundError('Set FULL_CSV, DATA_ZIP, or TRAIN_CSV and TEST_CSV explicitly.')
            paths = [trains[0], tests[0]]
    frames, sources = [], []
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(path)
        frame = pd.read_csv(path, low_memory=False)
        frames.append(frame)
        sources.append({'path': str(path), 'sha256': file_digest(path), 'rows': len(frame)})
    return combine_frames(frames), {'files': sources, 'combined_existing_splits': len(paths) == 2}

def stratified_three_way_split(y, seed):
    y = np.asarray(y)
    expected = set(range(len(REQUIRED_SEVERITY_LEVELS)))
    if set(np.unique(y)) != expected:
        raise ValueError('The full pool must include all four Severity levels (1–4).')
    counts = np.bincount(y, minlength=len(expected))
    if (counts < 3).any():
        raise ValueError('Each Severity level needs at least three records to appear in three partitions; more may be needed at these ratios.')
    positions = np.arange(len(y), dtype=np.int64)
    try:
        dev_idx, test_idx = train_test_split(
            positions, test_size=TEST_SIZE, stratify=y, random_state=seed)
        train_idx, val_idx = train_test_split(
            dev_idx, test_size=VAL_SIZE, stratify=y[dev_idx], random_state=seed)
    except ValueError as exc:
        raise ValueError('Too few records for stratified 64/16/20 splitting; add records for rare classes.') from exc
    combined = np.concatenate([train_idx, val_idx, test_idx])
    if len(combined) != len(y) or len(np.unique(combined)) != len(y):
        raise RuntimeError('Partitions overlap or fail to cover the full pool.')
    for name, idx in [('Train', train_idx), ('Validation', val_idx), ('Test', test_idx)]:
        missing = expected - set(np.unique(y[idx]))
        if missing:
            levels = [REQUIRED_SEVERITY_LEVELS[c] for c in sorted(missing)]
            raise ValueError(f'{name} is missing Severity {levels}. Add more rare-class records; splitting cannot create samples.')
    return train_idx, val_idx, test_idx

raw, input_identity = locate_full_pool(INPUT_ROOT)
original_rows = len(raw)
df = engineer_features(raw)
del raw
labels = pd.to_numeric(df[TARGET], errors='raise')
if not np.isfinite(labels).all() or not labels.isin(REQUIRED_SEVERITY_LEVELS).all():
    raise ValueError('Severity must contain only 1, 2, 3, or 4 after excluding missing labels.')
classes = sorted(labels.astype(int).unique().tolist())
missing_levels = sorted(set(REQUIRED_SEVERITY_LEVELS) - set(classes))
if missing_levels:
    raise ValueError(f'Full pool is missing Severity {missing_levels}; all four levels are required.')
class_map = {int(c): i for i, c in enumerate(classes)}
y = labels.astype(int).map(class_map).to_numpy(dtype=np.int64)
n_classes = len(classes)
X = df.drop(columns=[TARGET])
row_ids = X.index.to_numpy(dtype=np.int64)
del df, labels
gc.collect()
train_idx, val_idx, test_idx = stratified_three_way_split(y, RANDOM_STATE)
X_fit, X_val, X_test = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
y_fit, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
# Vocabulary/cardinality decisions are based on training rows only.
numeric_cols = [c for c in X_fit if pd.api.types.is_numeric_dtype(X_fit[c])]
cat_cols = [c for c in X_fit if c not in numeric_cols]
high_card = [c for c in cat_cols if X_fit[c].nunique() > MAX_CAT_ONEHOT]
low_card = [c for c in cat_cols if c not in high_card]
severity_counts = pd.DataFrame({
    name: np.bincount(part, minlength=n_classes)
    for name, part in [('Train', y_fit), ('Validation', y_val), ('Test', y_test)]
}, index=pd.Index(classes, name='Severity'))
if not (severity_counts.to_numpy() > 0).all():
    raise RuntimeError('Every partition must contain all four Severity levels.')
display(severity_counts)
severity_counts.to_csv(OUT_DIR / 'split_severity_counts.csv')
np.savez_compressed(OUT_DIR / 'split_row_ids.npz',
                    train=row_ids[train_idx], validation=row_ids[val_idx], test=row_ids[test_idx])
manifest = {'method': 'stratified two-stage holdout', 'seed': RANDOM_STATE,
            'test_fraction_of_full_pool': TEST_SIZE, 'validation_fraction_of_development': VAL_SIZE,
            'planned_fractions': {'train': (1-TEST_SIZE)*(1-VAL_SIZE),
                                  'validation': (1-TEST_SIZE)*VAL_SIZE, 'test': TEST_SIZE},
            'source': input_identity, 'original_rows': original_rows, 'usable_rows': len(y),
            'excluded_missing_label_rows': original_rows-len(y), 'class_map': class_map,
            'counts': {}, 'class_counts': {}, 'index_sha256': {}}
for name, idx in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]:
    manifest['counts'][name] = len(idx)
    manifest['class_counts'][name] = {str(classes[c]): int((y[idx] == c).sum()) for c in range(n_classes)}
    manifest['index_sha256'][name] = hashlib.sha256(np.sort(row_ids[idx]).tobytes()).hexdigest()
(OUT_DIR / 'split_manifest.json').write_text(json.dumps(manifest, indent=2))
joblib.dump(class_map, OUT_DIR / 'class_map.pkl')
print('Stratified Train/Validation/Test:', len(X_fit), len(X_val), len(X_test), '| classes:', class_map)
print('Actual percentages:', {name: round(100*count/len(y), 3) for name, count in manifest['counts'].items()})
print('Verified: Severity 1, 2, 3, and 4 are present in every partition.')
if set(GEO_COLS).issubset(X.columns):
    print('Missing coordinate rows retained:', int(X[GEO_COLS].isna().any(axis=1).sum()))
del X, y, row_ids
gc.collect()


## Shared preprocessing and metrics

In [ ]:
def calc_metrics(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro_Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}


THRESHOLD_BETA = 1.0

def tune_thresholds(y_val, proba_val, classes, beta=THRESHOLD_BETA):
    """Per-class decision thresholds maximizing F_beta on validation probabilities.
    Shared by LightGBM / XGBoost / RF / MLP / ST-GNN."""
    y_val = np.asarray(y_val); thresholds = {}
    for i, c in enumerate(classes):
        y_bin = (y_val == c).astype(int)
        prec, rec, thr = precision_recall_curve(y_bin, proba_val[:, i])
        thr = np.append(thr, 1.0)
        with np.errstate(divide="ignore", invalid="ignore"):
            fbeta = (1 + beta**2) * prec * rec / (beta**2 * prec + rec)
        fbeta = np.nan_to_num(fbeta)
        thresholds[c] = float(thr[np.argmax(fbeta)])
    return thresholds

def predict_with_thresholds(proba, thresholds, classes, fallback_class=0):
    """Apply per-class thresholds; samples passing no class fall back to fallback_class."""
    classes = np.array(classes)
    thr_arr = np.array([thresholds[c] for c in classes])
    passed = proba >= thr_arr[None, :]
    masked = np.where(passed, proba, -np.inf)
    preds = classes[masked.argmax(1)]
    preds[~passed.any(1)] = fallback_class
    return preds

def class_weight_dict(y, cap=CLASS_WEIGHT_CAP):
    """Balanced class weights as {class_label: weight}, capped so the ratio between the largest
    and smallest weight never exceeds `cap` (any weight above cap * min_weight is clipped down to
    that ceiling). This is the single source of truth for imbalance weighting - LightGBM, XGBoost,
    RandomForest (all via sample_weight), MLP (via oversampling below), and both GPU variants all
    derive their weighting from this same capped dict, so every model uses the identical ratio.
    Returns None if class weighting is off."""
    if not USE_CLASS_WEIGHT:
        return None
    classes = np.unique(y)
    raw_weights = compute_class_weight("balanced", classes=classes, y=y)
    min_w = raw_weights.min()
    capped_weights = np.minimum(raw_weights, min_w * cap)
    return dict(zip(classes, capped_weights))

def class_weights(y, cap=CLASS_WEIGHT_CAP):
    """Per-sample weights expanded from the capped class_weight_dict() above - used as
    `sample_weight` for LightGBM, XGBoost, and RandomForest (all trained via sample_weight rather
    than a "balanced"/"balanced_subsample" string, so the same cap applies to all three)."""
    cw = class_weight_dict(y, cap)
    if cw is None:
        return None
    y_arr = np.asarray(y)
    return np.array([cw[v] for v in y_arr])

def preprocess_ml(X_tr, y_tr, X_va, num=None, high=None, low=None):
    num = list(numeric_cols) if num is None else list(num)
    high = list(high_card) if high is None else list(high)
    low = list(low_card) if low is None else list(low)
    X_tr, X_va = X_tr.copy(), X_va.copy()
    original_columns = list(X_tr.columns)
    freq_maps = {}
    for c in high:
        f = X_tr[c].value_counts(normalize=True)
        freq_maps[c] = f
        X_tr[c+"_freq"] = X_tr[c].map(f).fillna(0.0).astype(np.float32)
        X_va[c+"_freq"] = X_va[c].map(f).fillna(0.0).astype(np.float32)
        num.append(c+"_freq")
    X_tr, X_va = X_tr.drop(columns=high), X_va.drop(columns=high)
    try:
        from sklearn.preprocessing import TargetEncoder
        enc = TargetEncoder(random_state=RANDOM_STATE)
    except Exception:
        from sklearn.preprocessing import OneHotEncoder
        enc = OneHotEncoder(handle_unknown="ignore", min_frequency=5)
    pre = ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")), ("e", enc)]), low),
    ])
    X_tr_p = pre.fit_transform(X_tr, y_tr)
    pre.high_card_freq_maps_ = freq_maps
    pre.high_card_columns_ = high
    pre.input_columns_ = original_columns
    return X_tr_p, pre.transform(X_va), pre


def dense32(a):
    return np.asarray(a.toarray() if sp.issparse(a) else a, dtype=np.float32)

def save_scores(name, model, Xval, Xtest, yval, ytest, out_dir, pre_labels, threshold_store,
                tuned_scores, argmax_scores, predictions):
    val_proba = np.asarray(model.predict_proba(Xval))
    test_proba = np.asarray(model.predict_proba(Xtest))
    classes = list(range(n_classes))
    thresholds = tune_thresholds(yval, val_proba, classes)
    majority = int(pd.Series(y_fit).mode()[0])
    tuned = predict_with_thresholds(test_proba, thresholds, classes, majority)
    argmax = test_proba.argmax(axis=1)
    threshold_store[name] = thresholds
    tuned_scores[name] = calc_metrics(ytest, tuned)
    argmax_scores[name] = calc_metrics(ytest, argmax)
    predictions[name] = np.asarray(pre_labels)[tuned]
    cm = confusion_matrix(ytest, tuned, labels=classes)
    pd.DataFrame(cm, index=pre_labels, columns=pre_labels).to_csv(out_dir / f'{name}_confusion_matrix.csv')
    pd.DataFrame(classification_report(ytest, tuned, labels=classes,
        target_names=[str(c) for c in pre_labels], output_dict=True, zero_division=0)).T.to_csv(
        out_dir / f'{name}_classification_report.csv')
    print(name, 'threshold-tuned TEST:', tuned_scores[name])

In [ ]:
Xtr_encoded, Xrest_encoded, pre = preprocess_ml(X_fit,y_fit,pd.concat([X_val,X_test]))
Xtr_np = dense32(Xtr_encoded)
Xval_np = dense32(Xrest_encoded[:len(X_val)])
Xtest_np = dense32(Xrest_encoded[len(X_val):])
joblib.dump(pre,OUT_DIR/'preprocess.pkl')
weights=class_weight_dict(y_fit)
weight_tensor = (torch.tensor([weights[c] for c in range(n_classes)],dtype=torch.float32,device=DEVICE)
                 if weights is not None else None)
thresholds, tuned_scores, argmax_scores, validation_scores = {},{},{},{}
predictions=pd.DataFrame({'Row_ID':X_test.index.to_numpy(dtype=np.int64), 'y_true':np.asarray(classes)[y_test]})

## MLP — mini-batches and validation early stopping

In [ ]:
class TorchMLP(nn.Module):
    def __init__(self,n_input,n_output):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_input,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(0.2),
                               nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(0.2),
                               nn.Linear(128,n_output))
    def forward(self,x):return self.net(x)

def mlp_probabilities(model,features,batch_size=4096):
    model.eval(); parts=[]
    with torch.no_grad():
        for first in range(0,len(features),batch_size):
            x=torch.from_numpy(features[first:first+batch_size]).to(DEVICE)
            parts.append(torch.softmax(model(x),dim=1).cpu().numpy())
    return np.concatenate(parts)

torch.manual_seed(RANDOM_STATE)
model=TorchMLP(Xtr_np.shape[1],n_classes).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
criterion=nn.CrossEntropyLoss(weight=weight_tensor)
train_loader=DataLoader(TensorDataset(torch.from_numpy(Xtr_np),torch.from_numpy(y_fit.astype(np.int64))),
                        batch_size=MLP_BATCH_SIZE,shuffle=True,pin_memory=True)
best_f1,best_state,bad=-1,None,0
for epoch in range(1,MLP_EPOCHS+1):
    model.train()
    for x,y in train_loader:
        if len(x)<2:continue
        x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(x),y)
        loss.backward();optimizer.step()
    val_proba=mlp_probabilities(model,Xval_np)
    score=f1_score(y_val,val_proba.argmax(axis=1),average='macro',zero_division=0)
    print(f'MLP epoch {epoch}: validation Macro_F1={score:.4f}')
    if score>best_f1:
        best_f1,bad=score,0
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else:
        bad+=1
        if bad>=MLP_PATIENCE:break
model.load_state_dict(best_state)
val_proba=mlp_probabilities(model,Xval_np)
test_proba=mlp_probabilities(model,Xtest_np)
validation_scores['MLP']=calc_metrics(y_val,val_proba.argmax(axis=1))
thresholds['MLP']=tune_thresholds(y_val,val_proba,list(range(n_classes)))
majority=int(pd.Series(y_fit).mode()[0])
tuned=predict_with_thresholds(test_proba,thresholds['MLP'],list(range(n_classes)),majority)
argmax=test_proba.argmax(axis=1)
tuned_scores['MLP']=calc_metrics(y_test,tuned)
argmax_scores['MLP']=calc_metrics(y_test,argmax)
predictions['MLP']=np.asarray(classes)[tuned]
pd.DataFrame(confusion_matrix(y_test,tuned,labels=list(range(n_classes))),
             index=classes,columns=classes).to_csv(OUT_DIR/'MLP_confusion_matrix.csv')
pd.DataFrame(classification_report(y_test,tuned,labels=list(range(n_classes)),
    target_names=[str(c) for c in classes],output_dict=True,zero_division=0)).T.to_csv(
    OUT_DIR/'MLP_classification_report.csv')
torch.save({'state_dict':model.state_dict(),'n_input':Xtr_np.shape[1],'n_classes':n_classes},OUT_DIR/'MLP.pt')
print('MLP test:',tuned_scores['MLP'])
del model,train_loader
torch.cuda.empty_cache()

## ST-GNN — full graph with neighbor-loader mini-batches

In [ ]:
class STGNN(nn.Module):
    def __init__(self,n_input,n_output,hidden=64):
        super().__init__()
        self.conv1=SAGEConv(n_input,hidden)
        self.conv2=SAGEConv(hidden,hidden)
        self.fc=nn.Linear(hidden,n_output)
    def forward(self,x,edge_index):
        x=F.relu(self.conv1(x,edge_index));x=F.dropout(x,p=0.5,training=self.training)
        x=F.relu(self.conv2(x,edge_index));x=F.dropout(x,p=0.5,training=self.training)
        return self.fc(x)

def build_graph(features):
    coords=features[GEO_COLS].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=np.float32)
    valid=np.flatnonzero(np.isfinite(coords).all(axis=1))
    if len(valid)<2:return torch.empty((2,0),dtype=torch.long)
    k=min(STGNN_K,len(valid)-1)
    neighbors=NearestNeighbors(n_neighbors=k+1,algorithm='ball_tree',n_jobs=-1).fit(coords[valid])
    ids=neighbors.kneighbors(coords[valid],return_distance=False)[:,1:]
    src=np.repeat(valid,k);dst=valid[ids.reshape(-1)]
    edge=np.empty((2,2*len(src)),dtype=np.int64)
    edge[0,:len(src)]=src;edge[1,:len(src)]=dst
    edge[0,len(src):]=dst;edge[1,len(src):]=src
    print('Graph nodes:',len(features),'directed edges:',edge.shape[1],
          'missing-location nodes retained:',len(features)-len(valid))
    return torch.from_numpy(edge)

all_raw=pd.concat([X_fit,X_val,X_test],ignore_index=True)
all_x=np.concatenate([Xtr_np,Xval_np,Xtest_np],axis=0)
all_y=np.concatenate([y_fit,y_val,y_test]).astype(np.int64)
n_tr,n_val=len(y_fit),len(y_val)
edge_index=build_graph(all_raw)
graph=Data(x=torch.from_numpy(all_x),edge_index=edge_index,y=torch.from_numpy(all_y))
graph.train_mask=torch.zeros(len(all_y),dtype=torch.bool);graph.train_mask[:n_tr]=True
graph.val_mask=torch.zeros(len(all_y),dtype=torch.bool);graph.val_mask[n_tr:n_tr+n_val]=True
graph.test_mask=torch.zeros(len(all_y),dtype=torch.bool);graph.test_mask[n_tr+n_val:]=True
train_loader=NeighborLoader(graph,input_nodes=graph.train_mask,num_neighbors=[10,10],
                            batch_size=GNN_BATCH_SIZE,shuffle=True)
val_loader=NeighborLoader(graph,input_nodes=graph.val_mask,num_neighbors=[10,10],
                          batch_size=GNN_BATCH_SIZE,shuffle=False)
test_loader=NeighborLoader(graph,input_nodes=graph.test_mask,num_neighbors=[10,10],
                           batch_size=GNN_BATCH_SIZE,shuffle=False)

def graph_probabilities(model,loader,start,n_rows):
    result=np.empty((n_rows,n_classes),dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch=batch.to(DEVICE)
            logits=model(batch.x,batch.edge_index)[:batch.batch_size]
            indices=batch.n_id[:batch.batch_size].cpu().numpy()-start
            result[indices]=torch.softmax(logits,dim=1).cpu().numpy()
    return result

model=STGNN(all_x.shape[1],n_classes).to(DEVICE)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
criterion=nn.CrossEntropyLoss(weight=weight_tensor)
best_f1,best_state,bad=-1,None,0
for epoch in range(1,STGNN_EPOCHS+1):
    model.train()
    for batch in train_loader:
        batch=batch.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits=model(batch.x,batch.edge_index)[:batch.batch_size]
        loss=criterion(logits,batch.y[:batch.batch_size]);loss.backward();optimizer.step()
    val_proba=graph_probabilities(model,val_loader,n_tr,n_val)
    score=f1_score(y_val,val_proba.argmax(axis=1),average='macro',zero_division=0)
    print(f'ST-GNN epoch {epoch}: validation Macro_F1={score:.4f}')
    if score>best_f1:
        best_f1,bad=score,0
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else:
        bad+=1
        if bad>=STGNN_PATIENCE:break
model.load_state_dict(best_state)
val_proba=graph_probabilities(model,val_loader,n_tr,n_val)
test_proba=graph_probabilities(model,test_loader,n_tr+n_val,len(y_test))
validation_scores['ST-GNN']=calc_metrics(y_val,val_proba.argmax(axis=1))
thresholds['ST-GNN']=tune_thresholds(y_val,val_proba,list(range(n_classes)))
tuned=predict_with_thresholds(test_proba,thresholds['ST-GNN'],list(range(n_classes)),majority)
argmax=test_proba.argmax(axis=1)
tuned_scores['ST-GNN']=calc_metrics(y_test,tuned)
argmax_scores['ST-GNN']=calc_metrics(y_test,argmax)
predictions['ST-GNN']=np.asarray(classes)[tuned]
pd.DataFrame(confusion_matrix(y_test,tuned,labels=list(range(n_classes))),
             index=classes,columns=classes).to_csv(OUT_DIR/'ST-GNN_confusion_matrix.csv')
pd.DataFrame(classification_report(y_test,tuned,labels=list(range(n_classes)),
    target_names=[str(c) for c in classes],output_dict=True,zero_division=0)).T.to_csv(
    OUT_DIR/'ST-GNN_classification_report.csv')
torch.save({'state_dict':model.state_dict(),'n_input':all_x.shape[1],'n_classes':n_classes,
            'graph_k':STGNN_K},OUT_DIR/'ST-GNN.pt')
print('ST-GNN test:',tuned_scores['ST-GNN'])

## Saved comparison files

In [ ]:
pd.DataFrame(validation_scores).T[METRICS].to_csv(OUT_DIR/'validation_summary.csv')
pd.DataFrame(tuned_scores).T[METRICS].to_csv(OUT_DIR/'test_comparison_matrix.csv')
pd.DataFrame(argmax_scores).T[METRICS].to_csv(OUT_DIR/'test_comparison_matrix_argmax_only.csv')
predictions.to_csv(OUT_DIR/'test_predictions.csv',index=False)
joblib.dump(thresholds,OUT_DIR/'class_thresholds.pkl')
print(pd.DataFrame(tuned_scores).T[METRICS].sort_values('Macro_F1',ascending=False))
print('Saved to:',OUT_DIR.resolve())